# CAMELYON16 + 17 → embedding H-optimus-0 su Colab

Esegue `td task=stream` su tutti gli 899 vetrini di sviluppo (399 CAMELYON16, 500 CAMELYON17). Per ogni vetrino: lo scarica sul disco della VM, calcola maschera tessuto, QC e griglia di tile, estrae gli embedding sulla GPU, li salva su **Google Drive** e cancella il vetrino.

- **Riprendibile.** Colab si disconnette (al massimo ~12 h, prima se la scheda resta inattiva): basta rieseguire tutte le celle, i vetrini già su Drive vengono saltati senza riscaricarli.
- **Spazio.** Circa 26 GB di embedding in float16 (8,35 milioni di tile × 1536 dimensioni) più tile e QC: la cella 2 controlla lo spazio libero su Drive.
- **Tempo.** Ordine di grandezza: qualche decina di ore di T4, quindi più sessioni. Il tempo vero si legge dalla cella 6 dopo i primi vetrini.
- **Solo dati pubblici.** Nessuna immagine clinica va caricata su Colab o su un Drive personale.

**Prima di iniziare**
- Runtime → Cambia tipo di runtime → **GPU T4**.
- Accetta i termini su https://huggingface.co/bioptimus/H-optimus-0 e salva un token di lettura come secret `HF_TOKEN` (icona della chiave a sinistra, *Notebook access* attivo).

In [ ]:
# 1. Hardware: GPU, CPU e disco locale (i vetrini passano di qui)
!nvidia-smi -L
!nproc
!df -h /content | tail -1

In [ ]:
# 2. Google Drive: gli embedding finiscono qui, così sopravvivono alle disconnessioni
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/tumordetector")
OUTS = {"camelyon16": OUT, "camelyon17": OUT / "camelyon17"}   # stessa disposizione di data/ sul Mac
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("/content/raw")
free_gb = shutil.disk_usage(OUT).free / 1e9
print(f"spazio libero su Drive: {free_gb:.0f} GB (servono circa 30 GB)")
assert free_gb > 30, "spazio insufficiente su Drive"

In [ ]:
# 3. Codice (se il repository è privato, clonalo con un token o rendilo pubblico)
REPO_URL = "https://github.com/masnarivan/TumorDetector.git"
REV = "origin/main"    # o un commit preciso; origin/ perché alla seconda sessione il clone esiste già e va aggiornato
!git clone -q {REPO_URL} /content/TumorDetector 2>/dev/null || git -C /content/TumorDetector fetch -q
!git -C /content/TumorDetector checkout -q {REV}
# Il progetto dichiara Python 3.12; Colab può essere più avanti (3.13 a ottobre 2026)
!pip install -q --ignore-requires-python -e /content/TumorDetector
# Le librerie che decidono maschera, griglia e pixel delle tile alle versioni di uv.lock: con quelle
# preinstallate su Colab la griglia può cambiare (e scikit-image < 0.26 non ha max_size)
!pip install -q scikit-image==0.26.0 scipy==1.18.1 pillow==12.3.0 shapely==2.2.0 \
    openslide-python==1.4.6 openslide-bin==4.0.1.2 timm==1.0.30
%cd /content/TumorDetector

In [ ]:
# 4. Pesi di H-optimus-0 (modello ad accesso controllato), con il token dal secret HF_TOKEN
import os
from huggingface_hub import snapshot_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("secret HF_TOKEN non disponibile:", type(e).__name__)
    from huggingface_hub import notebook_login
    notebook_login()
snapshot_download("bioptimus/H-optimus-0")

In [ ]:
# 5. Estrazione: prima CAMELYON16 poi CAMELYON17 (data.slides=null = tutti i vetrini del manifest).
#    Per una prova veloce: SLIDES = {"camelyon16": "[tumor_031,normal_108]"}
SLIDES = {"camelyon16": "null", "camelyon17": "null"}
# Velocità, non cambiano gli embedding. T4: 64 / 4; L4: 128 / vCPU; A100: 256 / 8 (non più dei vCPU di nproc)
BATCH, THREADS = 256, 8
PREFETCH = 2    # vetrini scaricati in parallelo: un solo download da S3 fa ~30 MB/s, meno di quanto elabora un'A100
for ds, slides in SLIDES.items():
    !td task=stream backbone=h_optimus_0 data={ds} data.raw_dir={RAW} paths.out={OUTS[ds]} \
        embed.batch_size={BATCH} embed.threads={THREADS} stream.prefetch={PREFETCH} \
        mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db "data.slides={slides}"

In [ ]:
# 6. Stato (eseguibile in qualsiasi momento, anche da una nuova sessione)
import json
import pandas as pd
total = {"camelyon16": 399, "camelyon17": 500}
counts = pd.read_csv("notebooks/tile_counts.csv").set_index("slide_id").n_tiles   # griglie del Mac
for ds, out in OUTS.items():
    root = out / "embeddings" / "h_optimus_0"
    if not root.exists():
        print(ds, "non ancora iniziato")
        continue
    print(ds, len(list(root.glob("*.npy"))), "/", total[ds], "vetrini con embedding")
    log = pd.read_csv(root / "extract_log.csv")   # ultima esecuzione
    done = log[log.status == "embedded"]
    if len(done):
        wall = done.t_wait_s.sum() + done.t_tile_s.sum() + done.t_embed_s.sum()
        print(f"  ultima esecuzione: {len(done)} vetrini, {done.n_tiles.sum():,.0f} tile, "
              f"{done.n_tiles.sum() / max(done.t_embed_s.sum(), 1):.0f} tile/s di embedding, "
              f"{wall / len(done) / 60:.1f} min per vetrino")
    grid = done.set_index("slide_id").n_tiles
    diff = grid[grid != counts.reindex(grid.index)]
    print("  griglia uguale a quella del Mac" if diff.empty else f"  ATTENZIONE, griglia diversa dal Mac: {diff.to_dict()}")
    bad = log[(log.status == "error") | log.status.str.startswith("referred")]
    if len(bad):
        display(bad)
print(f"spazio libero su Drive: {shutil.disk_usage(OUT).free / 1e9:.0f} GB")

Alla fine, sul Mac: copia `tumordetector/embeddings/h_optimus_0` da Drive in `data/embeddings/h_optimus_0` e `tumordetector/camelyon17/embeddings/h_optimus_0` in `data/camelyon17/embeddings/h_optimus_0` (le tile e il QC sono gli stessi già presenti in locale: la griglia non dipende dal backbone). Poi `td task=index backbone=h_optimus_0` per ciascun dataset.